# Fase A — Correcciones (v2)
Corrige lo que mostró la evaluación v1: el modelo usaba atajos (`https://www.`, URL corta sin ruta).
**Corrección 1** normalizar URLs · **2** legítimas con ruta · **3** prueba externa · **4** umbrales.
Los resultados v1 quedan en `models/` y `evidencias/`; los v2 van a `models_v2/` y `evidencias_v2/` para compararlos.

## Paso 0. Subir el proyecto actualizado
Sube `phishing_detector_v2.zip` (Archivos → subir) y ejecuta:

In [ ]:
!unzip -oq phishing_detector_v2.zip -d . 2>/dev/null; ls
%cd phishing_detector
!pip -q install ucimlrepo tf2onnx onnx onnxruntime
!python -c "import sys; sys.path.insert(0,'src'); from normalize import normalizar_url as n; print(n('HTTPS://www.Ejemplo.com/'))"   # debe imprimir: ejemplo.com

## RONDA 1 — Corrección 1 (normalizar) con el dataset que ya tienes
Si `data/raw/dataset_unificado.csv` ya no está (Colab reinició), vuelve a generarlo:

In [ ]:
import os
if not os.path.exists('data/raw/dataset_unificado.csv'):
    !python src/download_dataset.py --n-total 80000 --out data/raw/dataset_unificado.csv
else:
    print('Dataset ya presente')

In [ ]:
# 1.1 EDA (ahora alerta también por 'con_ruta')
!python src/eda.py --in data/raw/dataset_unificado.csv --out evidencias_v2

In [ ]:
# 1.2 Preprocesar CON normalización (sin esquema, sin www, sin '/' final)
!python src/preprocess.py --in data/raw/dataset_unificado.csv --outdir data/processed_v2 --maxlen 200

In [ ]:
# 1.3 Entrenar (20-45 min; si Colab corta, cambia a --k-folds 5)
!python src/train.py --processed-dir data/processed_v2 --models-dir models_v2 --epochs 15 --batch-size 128 --k-folds 10

In [ ]:
# 1.4 Evaluar: ahora mide con/sin ruta y sugiere umbrales
!python src/evaluate.py --model models_v2/cnn_phishing_final.keras --processed-dir data/processed_v2 --out evidencias_v2

### ¿Qué mirar?
* `recall` y `FPR` del TEST: es normal que bajen respecto a v1 (99.5 % / 0.03 %): ahora son más honestos.
* Bloque **"por tipo de URL"**: si `con_ruta` tiene `legitimas=0`, el FPR con ruta NO se puede medir → pasa a la **RONDA 2**.
* Si el FPR sube mucho o el recall baja de 0.85, pega esta salida en el chat.

## RONDA 2 — Corrección 2 (legítimas con ruta)
**Opción A (recomendada): Kaggle "Malicious URLs dataset".** Necesitas un token: kaggle.com → Settings → *Create New Token* → sube `kaggle.json`.

In [ ]:
# Sube kaggle.json (Archivos) y ejecuta:
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip -q install kaggle
!python src/download_dataset.py --n-total 80000 --descargar-kaggle --pct-legit-con-ruta 28 --out data/raw/dataset_v2.csv

**Opción B (sin Kaggle): entrenar solo con el host.** Es válida y honesta: la ruta deja de ser una pista que el dataset regala. Se pierde información de la ruta (limitación a declarar).

In [ ]:
# SOLO si no puedes usar Kaggle:
!python src/preprocess.py --in data/raw/dataset_unificado.csv --outdir data/processed_v2 --maxlen 200 --ruta-modo solo_host

In [ ]:
# Si usaste la Opción A, vuelve a correr preprocess/train/evaluate con el dataset nuevo:
!python src/eda.py --in data/raw/dataset_v2.csv --out evidencias_v2
!python src/preprocess.py --in data/raw/dataset_v2.csv --outdir data/processed_v2 --maxlen 200
!python src/train.py --processed-dir data/processed_v2 --models-dir models_v2 --epochs 15 --batch-size 128 --k-folds 10
!python src/evaluate.py --model models_v2/cnn_phishing_final.keras --processed-dir data/processed_v2 --out evidencias_v2

## Corrección 3 — Prueba EXTERNA (otra procedencia que el entrenamiento)
Phishing reciente de OpenPhish + legítimas de Tranco, sin repetir nada del entrenamiento.
*Limitación:* las legítimas de Tranco son solo dominios → no miden falsos positivos con ruta.

In [ ]:
!python src/construir_externo.py --train-csv data/raw/dataset_unificado.csv --out data/externo/externo.csv --n 1000
!python src/evaluate.py --model models_v2/cnn_phishing_final.keras --processed-dir data/processed_v2 --out evidencias_v2 --externo data/externo/externo.csv

## Corrección 4 — Umbrales
`evidencias_v2/umbrales_sugeridos.json` ya trae `umbral_advertencia` (recall ≥ 0.90 en validación) y `umbral_bloqueo` (FPR ≤ 0.5 %). Se calculan **después** de corregir el sesgo.

In [ ]:
!cat evidencias_v2/umbrales_sugeridos.json

## Exportar ONNX y latencia (v2)

In [ ]:
!python src/export_onnx.py --model models_v2/cnn_phishing_final.keras --out models_v2/cnn_phishing.onnx --processed-dir data/processed_v2

## Resumen para pegar en el chat y descargar

In [ ]:
import json, glob, os, zipfile
def lj(p): 
    return json.load(open(p)) if os.path.exists(p) else None
res = {
 "eda_alertas": (lj('evidencias_v2/eda_resumen.json') or {}).get('columnas_con_alerta'),
 "eda_atajos": (lj('evidencias_v2/eda_resumen.json') or {}).get('atajos_posibles'),
 "kfold_promedio": (lj('models_v2/cnn_phishing_kfold_metrics.json') or {}).get('promedio'),
 "test": lj('evidencias_v2/metricas_test.json'),
 "robustez": lj('evidencias_v2/robustez.json'),
 "umbrales": lj('evidencias_v2/umbrales_sugeridos.json'),
 "latencia": lj('models_v2/latencia_onnx_metrics.json'),
 "normalizacion": (lj('data/processed_v2/vocab_meta.json') or {}).get('normalizacion'),
}
print(json.dumps(res, indent=1, ensure_ascii=False))
with zipfile.ZipFile('fase_A_v2_resultados.zip','w',zipfile.ZIP_DEFLATED) as z:
    for d in ['evidencias_v2','models_v2']:
        for f in glob.glob(d+'/**/*', recursive=True):
            if os.path.isfile(f): z.write(f)
from google.colab import files; files.download('fase_A_v2_resultados.zip')